In [41]:
import numpy as np
import pandas as pd
import json
import sqlite3
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [42]:
## 1. Connect to the Database
conn = sqlite3.connect("../data/thanksgiving_point.db")
print("Database connection successful.")

Database connection successful.


In [43]:
## 2. Check available tables
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

tables

,name
0,children
1,households
2,survey_responses
3,visits


In [44]:
## 3. Extract households
households_df = pd.read_sql_query("SELECT * FROM households;",conn)

households_df

,household_id,family_size,adults,children_count,average_child_age
0,H00001,4,2,2,1.5
1,H00002,5,2,3,6.0
2,H00003,3,2,1,4.0
3,H00004,2,2,0,NaN
4,H00005,5,2,3,4.0
...,...,...,...,...,...
99995,H99996,6,2,4,6.0
99996,H99997,4,2,2,8.0
99997,H99998,2,2,0,NaN
99998,H99999,5,2,3,6.0


In [45]:
## 4. Extract children
children_df = pd.read_sql_query("SELECT * FROM children;", conn)
children_df

,child_id,household_id,age
0,H00001_C1,H00001,1
1,H00001_C2,H00001,2
2,H00002_C1,H00002,4
3,H00002_C2,H00002,6
4,H00002_C3,H00002,8
...,...,...,...
244137,H99999_C3,H99999,9
244138,H100000_C1,H100000,2
244139,H100000_C2,H100000,6
244140,H100000_C3,H100000,10


In [46]:
## 5. Extract visits
visits_df = pd.read_sql_query("SELECT * FROM visits;",conn)

visits_df

,visit_id,household_id,visit_date,visit_number,exhibit,crowd_density,number_of_programs,spending
0,H00001_V1,H00001,2026-01-18,1,Gardens,44.150495,4,39.63
1,H00001_V2,H00001,2026-09-03,2,Gardens,43.584904,3,71.70
2,H00001_V3,H00001,2026-09-06,3,Butterfly Biosphere,19.008313,2,63.19
3,H00001_V4,H00001,2026-07-15,4,Gardens,71.821010,0,66.13
4,H00001_V5,H00001,2026-05-15,5,Dinosaur Museum,35.720337,1,53.73
...,...,...,...,...,...,...,...,...
308892,H99998_V2,H99998,2026-03-02,2,Farm,53.604600,0,60.42
308893,H99998_V3,H99998,2026-01-26,3,Dinosaur Museum,67.325803,0,56.08
308894,H99998_V4,H99998,2026-06-11,4,Dinosaur Museum,62.652083,0,68.80
308895,H99999_V1,H99999,2026-08-24,1,Curiosity Museum,69.851207,0,75.38


In [47]:
## 6. Extract survey responses
survey_df = pd.read_sql_query("SELECT * FROM survey_responses;",conn)

survey_df

,survey_id,visit_id,household_id,satisfaction,learning_score,curiosity_score,confidence_score,social_interaction_score,return_intent,recommend_score,comments
0,H00001_V1_S1,H00001_V1,H00001,3.29,4.92,4.50,5.00,3.56,3,4,There were too many people in some exhibits.
1,H00001_V2_S1,H00001_V2,H00001,3.03,4.56,5.00,3.76,3.33,3,2,We learned a lot from the activity.
2,H00001_V3_S1,H00001_V3,H00001,3.96,3.45,4.23,4.48,4.40,5,3,The kids loved the experience.
3,H00001_V4_S1,H00001_V4,H00001,3.66,3.02,3.93,2.93,3.89,3,4,It would be better with fewer people.
4,H00001_V5_S1,H00001_V5,H00001,3.15,2.88,4.33,4.59,3.53,3,3,We enjoyed the hands-on learning activities.
...,...,...,...,...,...,...,...,...,...,...,...
308892,H99998_V2_S1,H99998_V2,H99998,5.00,3.49,4.67,4.38,3.04,4,5,We are looking forward to visiting again.
308893,H99998_V3_S1,H99998_V3,H99998,4.17,2.78,4.88,4.10,3.23,2,4,We really enjoyed our visit.
308894,H99998_V4_S1,H99998_V4,H99998,3.50,3.76,4.02,2.83,3.54,3,3,We are looking forward to visiting again.
308895,H99999_V1_S1,H99999_V1,H99999,4.99,2.63,3.89,4.04,4.14,4,4,The crowds made it harder for the kids to part...


In [48]:
## 8. Check the structure of each table
print("HOUSEHOLDS")
households_df.info()

print("\nCHILDREN")
children_df.info()

print("\nVISITS")
visits_df.info()

print("\nSURVEY RESPONSES")
survey_df.info()

HOUSEHOLDS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   household_id       100000 non-null  object 
 1   family_size        100000 non-null  int64  
 2   adults             100000 non-null  int64  
 3   children_count     100000 non-null  int64  
 4   average_child_age  93237 non-null   float64
dtypes: float64(1), int64(3), object(1)
memory usage: 3.8+ MB

CHILDREN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244142 entries, 0 to 244141
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   child_id      244142 non-null  object
 1   household_id  244142 non-null  object
 2   age           244142 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 5.6+ MB

VISITS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308897 entries, 0 to 308896
Data

In [49]:
## 9. Check missing values
print("Households missing values:")
display(households_df.isnull().sum())

print("Children missing values:")
display(children_df.isnull().sum())

print("Visits missing values:")
display(visits_df.isnull().sum())

print("Survey responses missing values:")
display(survey_df.isnull().sum())


Households missing values:


household_id            0
family_size             0
adults                  0
children_count          0
average_child_age    6763
dtype: int64

Children missing values:


child_id        0
household_id    0
age             0
dtype: int64

Visits missing values:


visit_id              0
household_id          0
visit_date            0
visit_number          0
exhibit               0
crowd_density         0
number_of_programs    0
spending              0
dtype: int64

Survey responses missing values:


survey_id                   0
visit_id                    0
household_id                0
satisfaction                0
learning_score              0
curiosity_score             0
confidence_score            0
social_interaction_score    0
return_intent               0
recommend_score             0
comments                    0
dtype: int64

In [50]:
## 10. Check relationships between tables
print("Household IDs:", households_df["household_id"].nunique())
print("Child household IDs:", children_df["household_id"].nunique())
print("Visit household IDs:", visits_df["household_id"].nunique())
print("Survey household IDs:", survey_df["household_id"].nunique())


Household IDs: 100000
Child household IDs: 93237
Visit household IDs: 100000
Survey household IDs: 100000
